# ReFuelEU optimisation — results

Reproduces the figures and Table 3 of the paper from the scenario dataset in `dataset/`
(described in its README), without running any optimisation: the notebook runs in under a
minute. After new runs, the dataset is rebuilt by the last cell of
`01_optimisation_runs.ipynb`.

Figures are written to `figures/`, under the file names the manuscript uses:

| paper | file | scenarios drawn |
|---|---|---|
| Figure 4 | `supply.pdf` | drawn in `equilibriums.ipynb`, which runs its own models |
| Figure 7 | `ressource_constraints.pdf` | the reference optimum, ReFuelEU |
| Figure 8 | `fig_sensitivity_summary.pdf` | blocks A–D, ladder rungs, other cases at the central budget |
| Figure 9 | `overall_layout2.pdf` | the `main` budget ladder and its references |
| Figure 10 | `2d_shares.pdf` | the ladders of the four biomass cases, ReFuelEU |
| Figure 11 | `sensitivity_curves.pdf` | fuel prices and emission factors of two runs |

In [ ]:
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import MaxNLocator
from scipy.interpolate import griddata

from helpers import dataset as D
from helpers import paper_style as style
from helpers.paper_style import (
    BAU,
    DIVERGING,
    GREEN,
    GREY,
    GRID,
    INK,
    LABEL_SIZE,
    MUTED,
    NAVY,
    NEUTRAL,
    OCHRE,
    ORANGE,
    PALE,
    PURPLE,
    RED,
    SEQUENTIAL_NAVY,
    SMALL_SIZE,
    TEAL,
)

style.apply()
pd.set_option("display.max_rows", 100)
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

YEARS = np.array(D.YEARS)
OPTIM_YEARS = D.OPTIM_YEARS
EU_ASK_SHARE = D.EU_ASK_SHARE
# The budget ladder of each case, tightest first: min CO2, then % of the world budget.
LADDER = ["mincarb", "2_0", "2_2", "2_4", "2_6", "2_8", "3_0", "3_2", "3_4", "3_6", "3_8"]
# 2019 airfare the demand function is anchored on (equilibriums.ipynb).
AIRFARE_2019 = 0.09236379319842411
# ReFuelEU at its milestones 2030-2050, split into the two pathways.
REFUELEU = {"biofuel": [4.8, 15.0, 24.0, 27.0, 35.0], "electrofuel": [1.2, 5.0, 10.0, 15.0, 35.0]}
# Primary resource per unit of fuel energy (data/energy_rte.yaml).
RESOURCE_PER_FUEL = {"biomass": 2.104355, "electricity": 2.290426}
CONSTRAINTS = {
    "blend_completeness_constraint": "Blend\ncompleteness",
    "biomass_trajectory_constraint": "Biomass\navailability",
    "electricity_trajectory_constraint": "Electricity\navailability",
    "biofuel_use_growth_constraint": "Biofuel\nramp-up",
    "electrofuel_use_growth_constraint": "Electrofuel\nramp-up",
}


def run(name):
    """Yearly outputs of one scenario, 2000-2050, one column per variable."""
    return D.load(name)[0]


def budget(name):
    """Share of the world carbon budget the scenario consumes (%)."""
    return D.scenarios().loc[name, "carbon_budget_consumed_share"] / EU_ASK_SHARE


def surplus_loss(name):
    """The objective: cumulative 2025-2050 surplus loss, discounted to 2020 (Bn EUR)."""
    return run(name)["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9

## Exploring the dataset

Every scenario is a name, listed with its parameters by `D.scenarios()`; the naming is in
`dataset/README.md`. For one scenario, `D.load(name)` returns its yearly outputs (every
variable of `dataset/variables.csv`) and `D.read_constraints(name)` its design variables
and constraint values at the optimum.

The table below gives the main results of all 78 scenarios. Surplus losses are measured
against 2019 unit economics, so only differences between scenarios are policy costs; the
undiscounted one sums the yearly loss over 2025–2050. Fixed demand has no demand curve,
hence no surplus.

In [ ]:
def key_results(name):
    v = run(name)
    surplus = "area_loss" in v
    return {
        "Airfare 2050 (€/RPK)": v["airfare_per_rpk"].loc[2050],
        "RPK CAGR 2025-2050 (%)": ((v["rpk"].loc[2050] / v["rpk"].loc[2025]) ** (1 / 25) - 1) * 100,
        "Biofuel 2050 (%)": v["generic_biofuel_share_dropin_fuel"].loc[2050],
        "Electrofuel 2050 (%)": v["generic_electrofuel_share_dropin_fuel"].loc[2050],
        "2050 AAF blend (%)": v["generic_biofuel_share_dropin_fuel"].loc[2050]
        + v["generic_electrofuel_share_dropin_fuel"].loc[2050],
        "Cumul. surplus loss vs Ref (Bn€)": (
            (v["area_loss"] + v["total_airline_cost_increase"]).loc[2025:2050].sum() / 1e9
            if surplus
            else np.nan
        ),
        "idem, discounted (Bn€)": surplus_loss(name) if surplus else np.nan,
        "CO2, 2050 (Mt)": v["co2_emissions_including_energy"].loc[2050],
        "Cumul. CO2, 2020-2050 (Gt)": v["cumulative_co2_emissions"].loc[2050],
        "Max. biomass consumption (EJ)": (
            v["generic_biofuel_energy_consumption"].max() * RESOURCE_PER_FUEL["biomass"] / 1e12
        ),
        "Max. electricity consumption (EJ)": (
            v["generic_electrofuel_energy_consumption"].max()
            * RESOURCE_PER_FUEL["electricity"]
            / 1e12
        ),
    }


results = pd.DataFrame({name: key_results(name) for name in D.scenarios().index}).T
results = D.scenarios()[["family", "label", "feasible"]].join(results)
results.round(3)

### Table 3 — key indicators across scenarios

The 2019 reference applies 2019 energy intensity, load factor and emission factor to the
exogenous traffic: it is the same in every run (`rpk_no_elasticity` and
`co2_emissions_last_historical_year_technology_baseline3`).

In [ ]:
reference = run("fossil_main")
table_3 = results.loc[
    ["fossil_main", "refueleu_main", "refueleu_step", "opt_main_refueleu"], results.columns[3:]
].T.drop(["Biofuel 2050 (%)", "Electrofuel 2050 (%)"])
table_3.columns = ["BAU", "ReFuelEU (linear)", "ReFuelEU (step)", "Optimisation"]
table_3.insert(
    0,
    "2019 Ref.",
    {
        "Airfare 2050 (€/RPK)": AIRFARE_2019,
        "RPK CAGR 2025-2050 (%)": (
            (reference["rpk_no_elasticity"].loc[2050] / reference["rpk_no_elasticity"].loc[2025])
            ** (1 / 25)
            - 1
        )
        * 100,
        "2050 AAF blend (%)": 0.0,
        "CO2, 2050 (Mt)": reference["co2_emissions_last_historical_year_technology_baseline3"].loc[
            2050
        ],
        "Cumul. CO2, 2020-2050 (Gt)": reference[
            "co2_emissions_last_historical_year_technology_baseline3"
        ]
        .loc[2020:2050]
        .sum()
        / 1e3,
        "Max. biomass consumption (EJ)": 0.0,
        "Max. electricity consumption (EJ)": 0.0,
    },
)
table_3.round(3)

## Figure 7 — the optimised mandate and the limits that shape it

`opt_main_refueleu`: the reference case optimised at ReFuelEU's own cumulative emissions
(3.87 GtCO₂, 3.12 % of the world budget). Panel a compares its mandate with ReFuelEU,
interpolated linearly between milestones and held in steps as legislated. Panels b and c
set each pathway's consumption against its two ramp-up limits (Eq. 12) and its resource
availability; red areas are infeasible.

In [ ]:
opt = run("opt_main_refueleu")
FUELS = {"generic_biofuel": GREEN, "generic_electrofuel": NAVY, "fossil_kerosene": RED}

optimum = opt[[f"{fuel}_share_dropin_fuel" for fuel in FUELS]].set_axis(list(FUELS), axis=1)
linear = pd.DataFrame(
    {
        "generic_biofuel": [0, 2, *REFUELEU["biofuel"]],
        "generic_electrofuel": [0, 0, *REFUELEU["electrofuel"]],
    },
    index=[2020, 2025, *OPTIM_YEARS],
)
step = pd.DataFrame(0.0, index=YEARS, columns=["generic_biofuel", "generic_electrofuel"])
for start, end, total, efuel in [
    (2025, 2029, 2, 0),
    (2030, 2034, 6, 1.2),
    (2035, 2039, 20, 5),
    (2040, 2044, 34, 10),
    (2045, 2049, 42, 15),
    (2050, 2050, 70, 35),
]:
    step.loc[start:end] = [total - efuel, efuel]
for mandate in (linear, step):
    mandate["fossil_kerosene"] = 100 - mandate["generic_biofuel"] - mandate["generic_electrofuel"]

fig, (ax, ax_bio, ax_ele) = plt.subplots(1, 3, figsize=(15, 5))
for mandate, ls, lw in [(optimum, "-", 2), (linear, "--", 1.5), (step, ":", 1.5)]:
    for fuel, colour in FUELS.items():
        ax.plot(mandate.index, mandate[fuel], ls=ls, color=colour, lw=lw)
ax.legend(
    handles=[
        mlines.Line2D([], [], color=colour, lw=1.5, label=label)
        for label, colour in [("Biofuel", GREEN), ("Electrofuel", NAVY), ("Fossil", RED)]
    ]
    + [
        mlines.Line2D([], [], color="k", lw=1.5, ls=ls, label=label)
        for label, ls in [
            ("Optimisation", "-"),
            ("ReFuelEU (linear)", "--"),
            ("ReFuelEU (step)", ":"),
        ]
    ],
    loc="center left",
)
ax.set_ylabel("Drop-in fuel shares (%)")
ax.set_xlim(2021, 2050)
ax.set_ylim(0, 105)
ax.set_title("a. Blending mandate")

for ax, title, pathway, resource, colour in [
    (ax_bio, "b. Biofuel", "generic_biofuel", "biomass", GREEN),
    (ax_ele, "c. Electrofuel", "generic_electrofuel", "electricity", NAVY),
]:
    consumption = opt[f"{pathway}_energy_consumption"]
    resource_cap = (
        opt[f"{resource}_availability_aviation_allocated"] / RESOURCE_PER_FUEL[resource] / 1e12
    )
    years = [2025]
    rate_caps = [consumption.loc[2025] / 1e12]
    volume_caps = [consumption.loc[2025] / 1e12]
    for year in OPTIM_YEARS:
        previous = consumption.loc[year - 5]
        years += [year, year]
        rate_caps += [previous * (1 + 0.2) ** 5 / 1e12, consumption.loc[year] / 1e12]
        volume_caps += [
            (previous + 0.2 * EU_ASK_SHARE * 5 * 1e12) / 1e12,
            consumption.loc[year] / 1e12,
        ]
    caps = np.maximum(rate_caps, volume_caps)

    ax.plot(YEARS, consumption / 1e12, color=colour, lw=3, label="Actual consumption")
    ax.plot(years, rate_caps, ":", color=PALE, lw=3, label="Ramp-up (rate)")
    ax.plot(years, volume_caps, "--", color=PALE, lw=2, label="Ramp-up (volume)")
    ax.plot(YEARS, resource_cap, "-", color=PALE, lw=2, label="Resource constraint")
    ax.fill_between(YEARS, resource_cap, 3, facecolor=RED, alpha=0.1, lw=0)
    ax.fill_between(years, caps, 3, facecolor=RED, alpha=0.1, lw=0)
    ax.fill_between(
        years,
        0,
        np.minimum(caps, np.interp(years, YEARS, resource_cap)),
        facecolor=GREEN,
        alpha=0.1,
        lw=0,
    )
    ax.set_ylabel("Consumption (EJ)")
    ax.set_xlim(2025, 2050)
    ax.set_ylim(-0.05, 1.5)
    ax.legend(loc="upper left")
    ax.set_title(title)

fig.tight_layout()
fig.savefig(FIGURES / "ressource_constraints.pdf")

## Figure 8 — sensitivity of the optimised mandate

One hypothesis changed per row, against `base` (the same problem as `opt_main_refueleu`,
reached from another start point; both land on the same optimum). Elasticity, ramp-up,
electrofuel pathway and discount rate are blocks A–D; the carbon budget rows are rungs of
the `main` ladder; the biomass and technology rows are the other cases re-optimised at the
central budget. In the mandate panels the ring is 2035 and the dot 2050. Surplus loss is
discounted at each run's own rate, so the † rows are not comparable in level. Drawn at its
printed size, 190 mm wide.

In [ ]:
GROUPS = [
    (
        "Carbon budget",
        NAVY,
        [
            ("opt_main_3_8", "3.8 %"),
            ("opt_main_3_4", "3.4 %"),
            ("opt_main_3_0", "3.0 %"),
            ("opt_main_2_6", "2.6 %"),
            ("opt_main_2_2", "2.2 %"),
        ],
    ),
    (
        "Biomass allocation",
        GREEN,
        [("opt_B5_refueleu", "5 %"), ("opt_B75_refueleu", "7.5 %"), ("opt_B15_refueleu", "15 %")],
    ),
    ("Technology", BAU, [("opt_pess_refueleu", "efficiency +0.91 %/yr")]),
    (
        "Price elasticity",
        PURPLE,
        [
            ("fixed_demand", "fixed demand"),
            ("eps_m0_6", "\u22120.6"),
            ("eps_m0_8", "\u22120.8"),
            ("eps_m1_0", "\u22121.0"),
            ("eps_m1_4", "\u22121.4"),
        ],
    ),
    (
        "Ramp-up rate",
        RED,
        [("rate_11_8", "11.8 %/yr  (IEA NZE)"), ("rate_39", "39 %/yr  (wind/PV)")],
    ),
    ("Ramp-up volume", ORANGE, [("vol_0_1", "0.1 EJ/yr"), ("vol_0_4", "0.4 EJ/yr")]),
    ("Electrofuel pathway", TEAL, [("efuel_wind", "dedicated wind")]),
    ("Discount rate", OCHRE, [("r_3_2", "3.2 % †"), ("r_7", "7 % †"), ("r_15", "15 % †")]),
]
PANELS = [
    ("a. Biofuel", "Mandate share (%)", ["2035", "2050"]),
    ("b. Electrofuel", "Mandate share (%)", ["2035", "2050"]),
    ("c. Surplus loss", "Loss (Bn €, discounted)", ["Cumulative to 2050"]),
    ("d. CO$_2$ emissions", "Emissions (MtCO$_2$/yr)", ["2050"]),
    ("e. Traffic", "Traffic (trillion RPK)", ["2050"]),
]
FONT = 7.0


def outcomes(name):
    v = run(name)
    return [
        tuple(v["generic_biofuel_mandate_share"].loc[[2035, 2050]]),
        tuple(v["generic_electrofuel_mandate_share"].loc[[2035, 2050]]),
        surplus_loss(name) if "cumulative_total_surplus_loss_discounted" in v else np.nan,
        v["co2_emissions_passenger"].loc[2050] + v["co2_emissions_freight"].loc[2050],
        v["rpk"].loc[2050] / 1e12,
    ]


baseline = outcomes("base")
rows, ticks, labels, label_styles, separators = (
    [(0.0, baseline, NEUTRAL)],
    [0.0],
    ["Baseline"],
    [(INK, "bold")],
    [],
)
y = 1.0
for group, colour, members in GROUPS:
    separators.append(y - 0.5)
    y += 0.5
    ticks.append(y)
    labels.append(group)
    label_styles.append((colour, "bold"))
    y += 1.0
    for name, label in members:
        ticks.append(y)
        labels.append(label)
        label_styles.append((INK, "normal"))
        rows.append((y, outcomes(name), colour))
        y += 1.0
mandate_top = max(max(values[i]) for _, values, _ in rows for i in (0, 1))

with plt.rc_context({"font.size": FONT, "legend.framealpha": 0}):
    fig = plt.figure(figsize=(190 / 25.4, 128 / 25.4), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.04, h_pad=0.02, wspace=0.03)
    grid = fig.add_gridspec(1, len(PANELS))
    axes = []
    for column in range(len(PANELS)):
        axes.append(fig.add_subplot(grid[0, column], sharey=axes[0] if axes else None))

    marker = {"ms": 4, "zorder": 2, "markeredgewidth": 0.9}
    ring = {**marker, "ms": 7, "color": "white"}
    for index, (ax, (title, xlabel, key)) in enumerate(zip(axes, PANELS)):
        style.tidy(ax, grid="x")
        ax.grid(axis="x", lw=0.5)
        ax.tick_params(labelsize=FONT, length=2.5, width=0.6, pad=2)
        ax.tick_params(axis="y", length=0, labelleft=index == 0)
        for boundary in separators:
            ax.axhline(boundary, color=GRID, lw=0.7, zorder=0)
        if index < 2:
            for reference in baseline[index]:
                ax.axvline(reference, color=MUTED, lw=0.7, ls=":", zorder=0)
            for y, values, colour in rows:
                for reference, value in zip(baseline[index], values[index]):
                    ax.plot(
                        [reference, value],
                        [y, y],
                        color=colour,
                        lw=0.9,
                        alpha=0.45,
                        solid_capstyle="butt",
                    )
                early, late = values[index]
                ax.plot(early, y, "o", markeredgecolor=colour, **ring)
                ax.plot(
                    late, y, "o", color=colour, markeredgecolor=colour, **{**marker, "zorder": 3}
                )
            ax.set_xlim(-4, max(mandate_top * 1.06, 63))
            ax.set_xticks(range(0, 61, 20))
        else:
            ax.axvline(baseline[index], color=MUTED, lw=0.7, ls=":", zorder=0)
            for y, values, colour in rows:
                ax.plot([baseline[index], values[index]], [y, y], color=colour, lw=0.9, alpha=0.45)
                ax.plot(values[index], y, "o", color=colour, markeredgecolor=colour, **marker)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))
        ax.set_xlabel(xlabel, fontsize=FONT, labelpad=2)
        key_styles = [
            {**ring, "markeredgecolor": MUTED},
            {**marker, "color": MUTED, "markeredgecolor": MUTED},
        ]
        ax.legend(
            [mlines.Line2D([], [], ls="none", marker="o", **s) for s in key_styles[-len(key) :]],
            key,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.075),
            ncol=len(key),
            frameon=False,
            fontsize=FONT,
            labelcolor=MUTED,
            handletextpad=0.1,
            columnspacing=1.2,
            borderpad=0,
            borderaxespad=0,
        )
        ax.set_title(title, fontsize=FONT + 1, pad=4)

    axes[0].set_yticks(ticks)
    axes[0].set_yticklabels(labels)
    for text, (colour, weight) in zip(axes[0].get_yticklabels(), label_styles):
        text.set_color(colour)
        text.set_fontweight(weight)
    axes[0].set_ylim(-0.9, ticks[-1] + 0.9)
    axes[0].invert_yaxis()

    fig.canvas.draw()
    boxes = [ax.get_position() for ax in axes]
    fig.set_layout_engine("none")
    width = boxes[0].width
    gap = (boxes[-1].x1 - boxes[0].x0 - len(axes) * width) / (len(axes) - 1)
    for index, (ax, box) in enumerate(zip(axes, boxes)):
        ax.set_position([boxes[0].x0 + index * (width + gap), box.y0, width, box.height])
    fig.savefig(FIGURES / "fig_sensitivity_summary.pdf", dpi=300)

## Figure 9 — dependence on the carbon budget

The `main` ladder, coloured by the share of the world carbon budget each run consumes,
against the BAU, ReFuelEU (linear) and the reference optimum. The 2.0 % target is
infeasible and ends at the minimum-CO₂ point, so the two share one tick. Panel f gives, for
each budget and enforcement year, the share of each limit the optimum uses (1 is the limit
itself); hatched rows have no feasible solution.

In [ ]:
ladder = {tag: run(f"opt_main_{tag}") for tag in LADDER}
ladder_budget = {tag: budget(f"opt_main_{tag}") for tag in LADDER}
bau, refueleu, reference = run("fossil_main"), run("refueleu_main"), run("opt_main_refueleu")
bar_min = min(ladder_budget.values()) - 0.05
norm = mcolors.Normalize(vmin=bar_min, vmax=max(ladder_budget.values()) + 0.1)
mark_at = [year - YEARS[0] for year in OPTIM_YEARS]

fig = plt.figure(figsize=(14, 13.5))
gs = fig.add_gridspec(3, 6, height_ratios=[1.25, 1.0, 1.35])
ax_bio = fig.add_subplot(gs[0, 0:3])
ax_ele = fig.add_subplot(gs[0, 3:6])
ax_co2 = fig.add_subplot(gs[1, 0:2])
ax_fare = fig.add_subplot(gs[1, 2:4])
ax_rpk = fig.add_subplot(gs[1, 4:6])
ax_heat = fig.add_subplot(gs[2, 0:6])

for ax, title, key, ylabel, ylim, scale in [
    (
        ax_bio,
        "a. Biofuel mandate",
        "generic_biofuel_share_dropin_fuel",
        "Biofuel share (%)",
        (-1, 59),
        1,
    ),
    (
        ax_ele,
        "b. Electrofuel mandate",
        "generic_electrofuel_share_dropin_fuel",
        "Electrofuel share (%)",
        (-1, 59),
        1,
    ),
    (
        ax_co2,
        "c. CO$_2$ emissions",
        "co2_emissions_including_energy",
        "Emissions (MtCO$_2$)",
        (0, None),
        1,
    ),
    (ax_fare, "d. Airfare", "airfare_per_rpk", "Airfare (€/RPK)", (0.06, 0.15), 1),
    (ax_rpk, "e. Traffic", "rpk", "Traffic (trillion RPK)", (0, None), 1e12),
]:
    for tag in LADDER:
        ax.plot(
            YEARS, ladder[tag][key] / scale, color=SEQUENTIAL_NAVY(norm(ladder_budget[tag])), lw=1.6
        )
    ax.plot(YEARS, bau[key] / scale, color=BAU, ls=":", lw=2.2)
    ax.plot(
        YEARS,
        refueleu[key] / scale,
        color=RED,
        ls="--",
        lw=2,
        marker="o",
        markevery=mark_at,
        ms=4.5,
    )
    ax.plot(
        YEARS,
        reference[key] / scale,
        color=GREEN,
        ls="--",
        lw=2,
        marker="s",
        markevery=mark_at,
        ms=4.5,
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    mandate = key.endswith("share_dropin_fuel")
    ax.set_xlim(2020 if mandate else 2019, 2050)
    ax.set_xticks(range(2020, 2051, 5 if mandate else 10))
    ax.set_ylim(*ylim)

ax_rpk.plot(YEARS, ladder["3_8"]["rpk_no_elasticity"] / 1e12, color="black", ls="--")
ax_fare.axhline(AIRFARE_2019, color="black", ls="--")
ax_bio.legend(
    handles=[
        mlines.Line2D([], [], color=BAU, ls=":", lw=2.2, label="BAU"),
        mlines.Line2D(
            [], [], color=RED, ls="--", lw=2, marker="o", ms=4.5, label="ReFuelEU (linear)"
        ),
        mlines.Line2D([], [], color=GREEN, ls="--", lw=2, marker="s", ms=4.5, label="Ref. optim."),
        mlines.Line2D([], [], color="black", ls="--", label="No elasticity / 2019 airfare"),
    ],
    loc="upper left",
)

heat_rows = LADDER[:0:-1] + ["mincarb"]
saturation = np.full((len(heat_rows), 5 * len(CONSTRAINTS) + 1), np.nan)
infeasible = []
for row, tag in enumerate(heat_rows):
    optimum = D.read_constraints(f"opt_main_{tag}")
    for block, name in enumerate(CONSTRAINTS):
        saturation[row, 5 * block : 5 * block + 5] = 1 + optimum["constraints"][name].to_numpy()
    saturation[row, -1] = 1 + optimum["carbon"]
    infeasible.append(not optimum["feasible"])

greys = plt.get_cmap("Greys")
active_mark = {
    "marker": "_",
    "color": "white",
    "ms": 9,
    "mew": 2,
    "path_effects": [pe.withStroke(linewidth=3.2, foreground="black")],
}
violated_mark = {
    "marker": "_",
    "color": RED,
    "ms": 9,
    "mew": 2.4,
    "path_effects": [pe.withStroke(linewidth=3.2, foreground="white")],
}
style.image_frame(ax_heat)
heat = ax_heat.imshow(saturation, aspect="auto", cmap=greys, vmin=0, vmax=1.25)
for (row, column), value in np.ndenumerate(saturation):
    if value > 1 + 1e-3:
        ax_heat.plot(column, row, **violated_mark)
    elif value >= 1 - 1e-3:
        ax_heat.plot(column, row, **active_mark)
for row, bad in enumerate(infeasible):
    if bad:
        ax_heat.add_patch(
            plt.Rectangle(
                (-0.5, row - 0.5),
                saturation.shape[1],
                1,
                fill=False,
                hatch="///",
                edgecolor=RED,
                lw=0,
            )
        )
for block, name in enumerate([*CONSTRAINTS.values(), "Carbon\nbudget"]):
    ax_heat.axvline(
        block * 5 - 0.5,
        color="white",
        lw=2.5,
        path_effects=[pe.withStroke(linewidth=5.0, foreground=NEUTRAL)],
    )
    ax_heat.text(
        min(block * 5 + 2, saturation.shape[1] - 1),
        -0.8,
        name,
        ha="center",
        va="bottom",
        fontsize=SMALL_SIZE,
    )

ax_heat.set_xticks(np.arange(saturation.shape[1]))
ax_heat.set_xticklabels(
    [str(year) for year in OPTIM_YEARS] * len(CONSTRAINTS) + ["all"],
    rotation=90,
    fontsize=SMALL_SIZE - 2,
)
ax_heat.set_xticks(np.arange(saturation.shape[1] + 1) - 0.5, minor=True)
ax_heat.set_yticks(np.arange(len(heat_rows) + 1) - 0.5, minor=True)
ax_heat.grid(which="minor", color="white", lw=0.8)
ax_heat.tick_params(which="minor", length=0)
ax_heat.set_yticks(np.arange(len(heat_rows)))
ax_heat.set_yticklabels(
    ["Min $CO_2$" if tag == "mincarb" else tag.replace("_", ".") for tag in heat_rows]
)
for label, bad in zip(ax_heat.get_yticklabels(), infeasible):
    if bad:
        label.set_color(RED)
ax_heat.set_ylabel("Share of world carbon budget (%)")
ax_heat.set_xlabel("Constraint enforcement year")
ax_heat.set_title("f. Binding constraints", pad=34)
active_cell = mpatches.Patch(facecolor=greys(1.0 / 1.25), edgecolor="none")
ax_heat.legend(
    [
        (active_cell, mlines.Line2D([], [], ls="none", **active_mark)),
        (active_cell, mlines.Line2D([], [], ls="none", **violated_mark)),
        mpatches.Patch(facecolor="white", edgecolor=RED, hatch="///"),
    ],
    ["Active", "Violated", "No feasible solution"],
    handler_map={tuple: HandlerTuple(ndivide=1)},
    handlelength=3.2,
    ncol=3,
    frameon=False,
    loc="lower right",
    bbox_to_anchor=(1.0, 1.16),
)

fig.subplots_adjust(left=0.07, right=0.85, top=0.95, bottom=0.06, hspace=0.55, wspace=0.75)
top, bottom = ax_bio.get_position().y1, ax_co2.get_position().y0
bar = fig.colorbar(
    plt.cm.ScalarMappable(cmap=SEQUENTIAL_NAVY, norm=norm),
    cax=fig.add_axes([0.885, bottom, 0.015, top - bottom]),
)
bar.ax.set_ylim(bar_min, budget("fossil_main") + 0.1)
bar.set_label("Share of world carbon budget (%)", labelpad=15)
ticked = sorted(
    (
        tag
        for tag in LADDER
        if tag == "mincarb" or abs(ladder_budget[tag] - ladder_budget["mincarb"]) >= 0.05
    ),
    key=ladder_budget.get,
)
bar.set_ticks([ladder_budget[tag] for tag in ticked])
bar.set_ticklabels(["Min $CO_2$" if tag == "mincarb" else tag.replace("_", ".") for tag in ticked])
bar.ax.tick_params(length=0, labelsize=SMALL_SIZE)
for tag in ticked:
    r, g, b, _ = SEQUENTIAL_NAVY(norm(ladder_budget[tag]))
    bar.ax.axhline(
        ladder_budget[tag],
        color="white" if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5 else "#222222",
        lw=0.8,
    )
for name, text, colour, ls, lw, offset in [
    ("refueleu_main", "ReFuelEU", RED, "--", 1.8, 0.015),
    ("opt_main_refueleu", "Ref. optim.", GREEN, (0, (2, 4)), 2.0, -0.035),
    ("fossil_main", "BAU", BAU, ":", 2.2, 0.0),
]:
    bar.ax.axhline(budget(name), color=colour, lw=lw, ls=ls)
    bar.ax.text(
        1.2,
        budget(name) + offset,
        text,
        color=colour,
        fontsize=SMALL_SIZE,
        va="center",
        ha="left",
        transform=bar.ax.get_yaxis_transform(),
    )

box = ax_heat.get_position()
heat_bar = fig.colorbar(heat, cax=fig.add_axes([0.885, box.y0, 0.015, box.height]))
heat_bar.set_label("Normalised constraint use")
heat_bar.ax.tick_params(labelsize=SMALL_SIZE)
fig.savefig(FIGURES / "overall_layout2.pdf", bbox_inches="tight")

## Figure 10 — surplus loss against carbon budget and biomass allocation

Every ladder run of `B5`, `B75`, `main` and `B15`, and the fossil BAU at each allocation,
is one point; the surface is interpolated through them. Runs missing their budget by more
than 1 % are left out, as they do not sit at the CO₂ they were asked for. Both ReFuelEU
readings are placed at their own CO₂ and biomass use and compared with the surface there.

In [ ]:
BIOMASS = {"B5": 5.0, "B75": 7.5, "main": 10.0, "B15": 15.0}
points = [
    (budget(name), biomass, surplus_loss(name))
    for case, biomass in BIOMASS.items()
    for name in [f"opt_{case}_{tag}" for tag in LADDER]
    if D.scenarios().loc[name, "aviation_carbon_budget_constraint"] <= 0.01
] + [(budget("fossil_main"), biomass, surplus_loss("fossil_main")) for biomass in BIOMASS.values()]
points = pd.DataFrame(points, columns=["co2", "biomass", "surplus"])

pad_x = 0.06 * (points["co2"].max() - points["co2"].min())
pad_y = 0.06 * (points["biomass"].max() - points["biomass"].min())
grid_x, grid_y = np.meshgrid(
    np.linspace(points["co2"].min() - pad_x, points["co2"].max() + pad_x, 200),
    np.linspace(points["biomass"].min() - pad_y, points["biomass"].max() + pad_y, 200),
)
grid_z = griddata(
    (points["co2"], points["biomass"]), points["surplus"], (grid_x, grid_y), method="cubic"
)
norm = mcolors.Normalize(vmin=-270, vmax=270)

fig, ax = plt.subplots(figsize=(10, 7))
mesh = ax.pcolormesh(grid_x, grid_y, grid_z, shading="gouraud", cmap=DIVERGING, norm=norm)
contours = ax.contour(grid_x, grid_y, grid_z, levels=15, colors="black", linewidths=0.8)
for label in ax.clabel(contours, inline=True, fontsize=LABEL_SIZE, fmt="%.1f", colors="black"):
    label.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])
fig.colorbar(mesh).set_label("Cumulative total surplus loss (Bn€, discounted)")
ax.scatter(
    points["co2"],
    points["biomass"],
    color=GREY,
    edgecolor="black",
    s=35,
    alpha=0.5,
    label="Scenario point",
    zorder=5,
)

for name, label, (dx, dy), (ha, va) in [
    ("refueleu_main", "ReFuelEU (linear)", (-0.06, 0.7), ("right", "bottom")),
    ("refueleu_step", "ReFuelEU (step)", (0.06, -0.7), ("left", "top")),
]:
    loss, co2 = surplus_loss(name), budget(name)
    biomass = run(name)["generic_biomass_consumed_global_share"].loc[2050] / EU_ASK_SHARE
    surface = griddata(
        (points["co2"], points["biomass"]), points["surplus"], (co2, biomass), method="cubic"
    )
    ax.scatter(
        co2,
        biomass,
        s=100,
        color=DIVERGING(norm(loss)),
        edgecolor="black",
        lw=1.2,
        zorder=10,
        label="ReFuelEU" if name == "refueleu_main" else None,
        path_effects=[pe.withStroke(linewidth=4, foreground="white")],
    )
    bold = label.replace(" ", r"\ ")
    ax.text(
        co2 + dx,
        biomass + dy,
        f"$\\bf{{{bold}}}$\n\u0394TS {loss:+.1f} Bn€\n\u0394 vs surface {loss - float(surface):+.1f} Bn€",
        fontsize=LABEL_SIZE - 1,
        linespacing=1.8,
        color=PURPLE,
        ha=ha,
        va=va,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor=PURPLE,
            linewidth=0.8,
            alpha=0.45,
        ),
        zorder=11,
    )

ax.set_xlabel("Carbon budget consumed share (%)")
ax.set_ylabel("Biomass allocated to aviation (%)")
ax.grid(True, ls="--")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "2d_shares.pdf")

## Figure 11 — carbon abatement cost against the discount rate

Fuel prices and emission factors only, no optimisation: grid electrofuel from
`refueleu_main`, dedicated-wind electrofuel from `efuel_wind`. Costs are discounted to
2020, tonnes are not, so a tonne abated late competes with a tonne abated early. The
marked crossing is the rate above which 2050 electrofuel is cheaper per tonne than 2025
biofuel.

In [ ]:
grid_prices, wind_prices = run("refueleu_main"), run("efuel_wind")
EIS = [2025, 2030, 2040, 2050]
BIO_RAMP = dict(zip(EIS, style.BIOFUEL_RAMP))
EFUEL_RAMP = dict(zip(EIS, style.ELECTROFUEL_RAMP))
RATES = (3.2, 7.0, 15.0)
ARTICLE_RATE = 4.5
HALO = [pe.withStroke(linewidth=2.5, foreground="white")]
rates = np.linspace(0.0, 0.2, 2001)


def abatement_cost(pathway, prices, year):
    """Undiscounted EUR/tCO2 against fossil kerosene, None where no cleaner than it."""
    delta_price = (
        prices[f"{pathway}_mean_mfsp"].loc[year]
        - grid_prices["fossil_kerosene_mean_mfsp"].loc[year]
    )
    delta_emissions = (
        grid_prices["fossil_kerosene_mean_co2_emission_factor"].loc[year]
        - prices[f"{pathway}_mean_co2_emission_factor"].loc[year]
    ) / 1e6
    return delta_price / delta_emissions if delta_emissions > 0 else None


PANELS = {
    "a": (
        "a. Electrofuel from grid electricity",
        grid_prices,
        {-1: "2025 biofuel cheaper", 1: "2050 e-fuel cheaper"},
    ),
    "b": ("b. Electrofuel from dedicated wind", wind_prices, {1: "2050 e-fuel cheaper"}),
}
YEAR_LABELS = {
    "a": [
        ("2030", 0.2, 2504, 1, 1.5, "left", "bottom", "efuel"),
        ("2040", 0.2, 969, 1, 1.5, "left", "bottom", "efuel"),
        ("2050", 6.2, 125, 3, 4.5, "left", "bottom", "efuel"),
        ("2025", 18.6, 162, -3, -4.5, "right", "top", "bio"),
        ("2030", 18.6, 69, -1, -1.5, "right", "top", "bio"),
        ("2040", 18.6, 12.5, -1, -1.5, "right", "top", "bio"),
        ("2050", 13.0, 9.7, 1, 1.5, "left", "bottom", "bio"),
    ],
    "b": [
        ("2025", 0.2, 1056, 1, 1.5, "left", "bottom", "efuel"),
        ("2030", 8.0, 363, 1, 1.5, "left", "bottom", "efuel"),
        ("2040", 12.0, 61, 1, 1.5, "left", "bottom", "efuel"),
        ("2050", 9.0, 34, 3, 4.5, "left", "bottom", "efuel"),
        ("2025", 18.2, 164, 3, 4.5, "left", "bottom", "bio"),
        ("2030", 18.6, 69, -1, -1.5, "right", "top", "bio"),
        ("2040", 18.6, 12.5, -1, -1.5, "right", "top", "bio"),
        ("2050", 12.6, 10.8, -1, -1.5, "right", "top", "bio"),
    ],
}

bio = {year: abatement_cost("generic_biofuel", grid_prices, year) for year in EIS}
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.8), sharey=True)
crossings = {}
for ax, (panel, (title, prices, _)) in zip(axes, PANELS.items()):
    efuel = {year: abatement_cost("generic_electrofuel", prices, year) for year in EIS}
    ax.axvspan(3.0, 7.0, color="0.9", lw=0, zorder=0)
    for rate in RATES:
        ax.axvline(
            rate,
            ymin=0.38 if (panel, rate) == ("a", 3.2) else 0,
            color="0.45",
            lw=0.8,
            ls=":",
            zorder=1,
        )
    ax.axvline(ARTICLE_RATE, color="0.25", lw=1, zorder=1)
    for rate in (*RATES, ARTICLE_RATE):
        ax.annotate(
            f"{rate:g} %",
            (rate, 3900),
            xytext=(2, 0),
            textcoords="offset points",
            rotation=90,
            ha="left",
            va="top",
            fontsize=8,
            color="0.15" if rate == ARTICLE_RATE else "0.3",
            path_effects=HALO,
            zorder=6,
        )
    for year in EIS:
        for cost, ramp, bold in (
            (bio[year], BIO_RAMP, year == 2025),
            (efuel[year], EFUEL_RAMP, year == 2050),
        ):
            if cost is not None:
                ax.plot(
                    rates * 100,
                    cost / (1 + rates) ** (year - 2020),
                    color=ramp[year],
                    lw=2.2 if bold else 1.4,
                    alpha=1 if bold else 0.3,
                    zorder=3 if bold else 2,
                )
    rate = (efuel[2050] / bio[2025]) ** (1 / 25) - 1
    value = bio[2025] / (1 + rate) ** 5
    crossings[panel] = (rate * 100, value)
    ax.plot([rate * 100] * 2, [0, value], color="0.15", lw=0.8, ls=(0, (2, 2)), zorder=4)
    ax.plot(rate * 100, value, "o", ms=6.5, mfc="0.15", mec="0.15", mew=1.0, zorder=5)
    ax.set_title(title, fontsize=10.5, pad=16)
    ax.set_yscale("log")
    ax.set_xlim(0, 20)
    ax.set_ylim(8, 4000)
    ax.set_yticks([10, 20, 50, 100, 200, 500, 1000, 2000])
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_yticks([], minor=True)
    ax.tick_params(labelsize=8)
    ax.set_xlabel("Discount rate (%)", fontsize=9)
axes[0].set_ylabel("Carbon abatement cost (€/tCO$_2$)\neuros discounted to 2020", fontsize=9)
fig.legend(
    [
        tuple(mlines.Line2D([], [], color=BIO_RAMP[year], lw=2.2) for year in EIS),
        tuple(mlines.Line2D([], [], color=EFUEL_RAMP[year], lw=2.2) for year in EIS),
    ],
    ["Biofuel, EIS 2025 \u2192 2050", "Electrofuel, EIS 2025 \u2192 2050"],
    handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
    handlelength=4.5,
    loc="lower center",
    ncol=2,
    frameon=False,
    fontsize=8,
)
fig.tight_layout(rect=(0, 0.07, 1, 1))

for ax, (panel, (_, _, regions)) in zip(axes, PANELS.items()):
    rate, value = crossings[panel]
    ax.annotate(
        f"{rate:.1f} %",
        (rate, value),
        xytext=(7, 6),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="0.1",
        path_effects=HALO,
        zorder=6,
    )
    for side, text in regions.items():
        ax.annotate(
            text,
            (rate, 9.5),
            xytext=(side * 3, 0),
            textcoords="offset points",
            rotation=90,
            ha="right" if side < 0 else "left",
            va="bottom",
            fontsize=8,
            color="0.25",
            linespacing=1.0,
            path_effects=HALO,
            zorder=6,
        )
    ax.annotate(
        "Article rate",
        (ARTICLE_RATE, 9.5),
        xytext=(4, 0),
        textcoords="offset points",
        rotation=90,
        ha="left",
        va="bottom",
        fontsize=8,
        color="0.15",
        linespacing=1.0,
        path_effects=HALO,
        zorder=6,
    )
    ax.text(
        5.0,
        1.01,
        "Typical social discount rates",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=8,
        color="0.3",
        linespacing=1.0,
    )
    for text, x, y, dx, dy, ha, va, kind in YEAR_LABELS[panel]:
        year = int(text)
        bold = year == (2050 if kind == "efuel" else 2025)
        ax.annotate(
            text,
            (x, y),
            xytext=(dx, dy),
            textcoords="offset points",
            ha=ha,
            va=va,
            fontsize=8,
            path_effects=HALO,
            zorder=6,
            color=(EFUEL_RAMP if kind == "efuel" else BIO_RAMP)[year],
            alpha=1 if bold else 0.8,
            fontweight="bold" if bold else "normal",
        )

fig.savefig(FIGURES / "sensitivity_curves.pdf")